# III. From tryptiques to movement

Objectif de ce notebook : maintenant que l'on a les tryptiques SUJET/VERBE/OBJET d'un texte en .csv, réutilisons les entités nommés stockées dans le .conllu correspondant pour avoir des tryptiques PERSONNAGE/VERBE/LIEU (ou dans l'autre sens).

### 1) Imports et chemins

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# CSV_PATH = "../results/csv_triptyques/triptyques_tdm.csv"
CSV_PATH = "../results/csv_triptyques/triptyques_chap1to5_sacr.csv"

# CONLLU_PATH = "../results/conllu_dynamouv/1872_Verne-Jules_Le-tour-du-monde-en-quatre-vingts-jours.conllu"
CONLLU_PATH = "../results/from_sacr/all_annots.sacr.conllu"

# OUT_PATH = "../results/exploration/PROPP/TDM_LIEU_PERSONNAGE.csv"
OUT_PATH = "../results/exploration/SACR/chap1to5_LIEU_PERSONNAGE.csv"

In [ ]:
df_all = pd.read_csv(CSV_PATH)
df_all = df_all.reset_index(drop=True)
df_all["triptyque_id"] = np.arange(1, len(df_all) + 1)

### 2) Lire le CONLLU

In [ ]:
# 2) Charger le CONLU
def parse_conllu(path):
    sentences = []
    comments = []
    tokens = []

    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.rstrip("\n")

            if not line.strip():
                if tokens:
                    sentences.append({"comments": comments, "tokens": tokens})
                comments, tokens = [], []
                continue

            if line.startswith("#"):
                comments.append(line)
                continue

            parts = line.split("\t")
            if len(parts) < 10:
                continue

            try:
                tok_id = int(parts[0])
                head = int(parts[6]) if parts[6] != "_" else 0
            except ValueError:
                continue

            tokens.append({
                "id": tok_id,
                "form": parts[1],
                "lemma": parts[2],
                "upos": parts[3],
                "head": head,
                "deprel": parts[7],
                "ent_text": parts[8],   # Propp : nom/coref de l'entité
                "ent_cat": parts[9],    # Propp : PER / FAC / LOC / etc.
            })

    if tokens:
        sentences.append({"comments": comments, "tokens": tokens})

    return sentences


sentences = parse_conllu(CONLLU_PATH)
sent_by_index = {i + 1: s["tokens"] for i, s in enumerate(sentences)}

### 3) Fonctions pour reconstruire le groupe sujet/objet


In [ ]:

def children_map(tokens):
    children = {}
    for tok in tokens:
        children.setdefault(tok["head"], []).append(tok)
    return children


def group_token_ids(head_id, tokens):
    """
    Reprend la logique du notebook 2 :
    groupe = tête + descendants non verbaux, hors ponctuation.
    """
    if pd.isna(head_id):
        return []

    head_id = int(head_id)
    by_id = {tok["id"]: tok for tok in tokens}
    children = children_map(tokens)
    collected = set()

    def rec(tok_id):
        if tok_id in collected or tok_id not in by_id:
            return

        tok = by_id[tok_id]

        if tok["upos"] == "VERB":
            return
        if tok["deprel"] == "punct":
            return

        collected.add(tok_id)

        for child in children.get(tok_id, []):
            rec(child["id"])

    rec(head_id)
    return sorted(collected)


def unique_list(x):
    out = []
    for v in x:
        if pd.notna(v) and v not in out and v != "_":
            out.append(v)
    return out


def entities_in_ids(tokens, ids, wanted_cats):
    texts = []
    cats = []

    ids = set(ids)

    for tok in tokens:
        if tok["id"] in ids and tok["ent_cat"] in wanted_cats:
            texts.append(tok["ent_text"])
            cats.append(tok["ent_cat"])

    return unique_list(texts), unique_list(cats)

In [10]:
# 4) Annoter chaque triptyque, pas chaque phrase

rows = []

for _, row in df_all.iterrows():
    sent_idx = int(row["Sentence_index"])
    tokens = sent_by_index.get(sent_idx, [])

    sujet_ids = group_token_ids(row["ID_sujet"], tokens)
    objet_ids = group_token_ids(row["ID_objet"], tokens)

    # Entités Propp dans le sujet
    pers_sujet, _ = entities_in_ids(tokens, sujet_ids, {"PER"})
    lieux_sujet, lieux_sujet_cat = entities_in_ids(tokens, sujet_ids, {"FAC", "LOC"})

    # Entités Propp dans l'objet
    pers_objet, _ = entities_in_ids(tokens, objet_ids, {"PER"})
    lieux_objet, lieux_objet_cat = entities_in_ids(tokens, objet_ids, {"FAC", "LOC"})

    personnages_triptyque = unique_list(pers_sujet + pers_objet)
    lieux_triptyque = unique_list(lieux_sujet + lieux_objet)
    types_lieux_triptyque = unique_list(lieux_sujet_cat + lieux_objet_cat)

    rows.append({
        "triptyque_id": row["triptyque_id"],

        "personnages_sujet": pers_sujet,
        "personnages_objet": pers_objet,
        "personnages_triptyque": personnages_triptyque,

        "lieux_sujet": lieux_sujet,
        "lieux_objet": lieux_objet,
        "lieux_triptyque": lieux_triptyque,
        "types_lieux_triptyque": types_lieux_triptyque,

        "a_personnage_triptyque": len(personnages_triptyque) > 0,
        "a_lieu_triptyque": len(lieux_triptyque) > 0,
        "a_personnage_et_lieu_triptyque": (
            len(personnages_triptyque) > 0 and len(lieux_triptyque) > 0
        ),
    })

ann = pd.DataFrame(rows)
df_trip_propp = df_all.merge(ann, on="triptyque_id", how="left")

In [11]:
# 5) Comptes demandés

print("Nombre total de triptyques :", len(df_trip_propp))
print("Nombre de triptyques avec personnage PER :", int(df_trip_propp["a_personnage_triptyque"].sum()))
print("Nombre de triptyques avec lieu FAC/LOC :", int(df_trip_propp["a_lieu_triptyque"].sum()))
print("Nombre de triptyques avec personnage ET lieu :", int(df_trip_propp["a_personnage_et_lieu_triptyque"].sum()))

Nombre total de triptyques : 1255
Nombre de triptyques avec personnage PER : 752
Nombre de triptyques avec lieu FAC/LOC : 88
Nombre de triptyques avec personnage ET lieu : 45


In [12]:
# 6) Export : seulement les triptyques avec personnage ET lieu

triptyques_pers_lieu = df_trip_propp[
    df_trip_propp["a_personnage_et_lieu_triptyque"]
].copy()


triptyques_pers_lieu.to_csv(
    OUT_PATH,
    index=False,
    encoding="utf-8"
)

print("CSV sauvegardé :", OUT_PATH)

triptyques_pers_lieu[[
    "Phrase",
    "Sujet",
    "Verbe",
    "Objet",
    "personnages_sujet",
    "personnages_objet",
    "lieux_sujet",
    "lieux_objet",
    "personnages_triptyque",
    "lieux_triptyque",
    "types_lieux_triptyque"
]].head(20)

CSV sauvegardé : ../results/exploration/SACR/chap1to5_LIEU_PERSONNAGE.csv


,Phrase,Sujet,Verbe,Objet,personnages_sujet,personnages_objet,lieux_sujet,lieux_objet,personnages_triptyque,lieux_triptyque,types_lieux_triptyque
0,"En l' année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville - row Burlington Garden...,[],"[5, 10]",[],[5],"[5, 10]",[5],[FAC]
41,À qui s' étonnerait de ce qu' un gentleman aus...,un gentleman aussi mystérieux,comptât,parmi les membres de cette honorable association,[0],[],[],[3],[0],[3],[FAC]
110,Phileas Fogg vivait seul dans sa maison de Sav...,Phileas Fogg,vivait,dans sa maison de Saville - row,[0],[0],[],"[5, 10]",[0],"[5, 10]",[FAC]
122,"Déjeunant, dînant au club à des heures chronom...",il,rentrait,chez lui,[0],[0],[],[5],[0],[5],[FAC]
130,"Sur vingt-quatre heures, il en passait dix à s...",il,passait,à son domicile,[0],[0],[],[5],[0],[5],[FAC]
136,"S' il se promenait, c' était invariablement, d...",il,promenait,dans la salle d' entrée ou sur la galerie circ...,[0],[],[],"[174, 172]",[0],"[174, 172]",[FAC]
189,"À onze heures et demie sonnant, Mr. Fogg devai...",Mr. Fogg,quitter,la maison,[0],[],[],[5],[0],[5],[FAC]
191,"À onze heures et demie sonnant, Mr. Fogg devai...",Mr. Fogg,rendre,au Reform,[0],[],[],[3],[0],[3],[FAC]
192,"À onze heures et demie sonnant, Mr. Fogg devai...",Mr. Fogg,rendre,-,[0],[],[],[3],[0],[3],[FAC]
193,"À onze heures et demie sonnant, Mr. Fogg devai...",Mr. Fogg,rendre,Club,[0],[],[],[3],[0],[3],[FAC]
